# 5 — middleware and booking

So far every question to the agent is answered on its own: ask a follow-up
and it has no idea what came before. And it can search, price and look up
policy, but it can't book anything. This book gives it a memory of the
conversation, then gives it a pen — the ability to write a reservation —
and then puts a hand on the pen, so nothing is booked until you say yes.

In [15]:
from hotelbot.config import configure_tracing
from hotelbot.agent import build_agent, run
from hotelbot.reservations import load_reservations, reset_reservations

configure_tracing(book=5)
reset_reservations()

agent = build_agent()

**Each call starts blank.** Ask the agent for hotels, then follow up with
"is the second one free a day later?" — a question that only makes sense if
it remembers its own list. Each `run()` call here starts a new conversation:
the model receives the system prompt and your one new message, nothing else.
Predict before running: does the second answer know which hotel "the second
one" is?

In [16]:
first = run(agent, "Two nights in Alfama, Lisbon, 10-12 Oct, under €150 — what is there?")
print(first.structured_response)
print()

second = run(agent, "Is the second one free a day later?")
print(second.structured_response)

text='For Oct 10–12 in Alfama, Lisbon (under €150/night), all three look available:\n\n1. Alfama House – €90/night, wifi, gym, pool (4.5★) → €180 total\n2. Alfama Loft – €125/night, wifi, breakfast (4.1★) → €250 total\n3. Casa do Castelo – €140/night, breakfast, wifi, gym (4.6★) → €280 total\n\nWant me to check exact availability or book one?'

text='Could you clarify which hotel ("the second one") and the dates you\'d like checked? It looks like I don\'t have the earlier search results in this conversation.'


**Memory is a checkpointer.** In book 1 you passed the model a list of
messages; it only knew what was in that list. A checkpointer keeps that list
for you. After each step of the loop it saves the conversation under a
*thread id* — any string you pick to name the conversation — and when the
next call arrives with the same thread id, it loads the saved messages and
appends the new one. `build_agent()` already comes with one, an
`InMemorySaver`, which keeps threads in a Python dict and forgets them when
the kernel stops.

`run()` makes up a fresh thread id when you don't give one, which is why the
cell above forgot. Run the same two turns with `thread_id="alfama"` on both,
then print every message the second call saw: your first question, the tool
calls and results that answered it, the first answer (a `PlainAnswer` tool
call, as in book 3), and only then the follow-up. Running this cell a second
time adds two more turns to the same thread, and the list grows.

In [17]:
print(type(agent.checkpointer).__name__)
print()

first = run(agent, "Two nights in Alfama, Lisbon, 10-12 Oct, under €150 — what is there?", thread_id="alfama")
second = run(agent, "Is the second one free a day later?", thread_id="alfama")
print(second.structured_response)
print()

for m in second.messages:
    detail = [call["name"] for call in m.tool_calls] if getattr(m, "tool_calls", None) else m.text[:70]
    print(f"{type(m).__name__:<13} {detail}")

InMemorySaver

text='Yes — Alfama Loft is available Oct 11–13 (2 nights), total €250.'

HumanMessage  Two nights in Alfama, Lisbon, 10-12 Oct, under €150 — what is there?
AIMessage     ['search_hotels']
ToolMessage   [{"id": "lis-001", "name": "Alfama House", "city": "Lisbon", "district
AIMessage     ['PlainAnswer']
ToolMessage   Returning structured response: text='For Alfama, Lisbon, Oct 10–12 (2 
HumanMessage  Is the second one free a day later?
AIMessage     ['check_availability']
ToolMessage   {"ok": true, "nights": 2, "total": 250.0, "reason": null}
AIMessage     ['PlainAnswer']
ToolMessage   Returning structured response: text='Yes — Alfama Loft is available Oc


**The write.** Every tool so far only read data. A reservation is the first
thing the project writes: `hotelbot.reservations.add_reservation` checks the
stay is available, gives it the next id (`res-0001`, `res-0002`, ...) and
appends it to `data/reservations.json`. Call it as plain Python — no model
involved — and print the file. Then reset it to empty, since the setup cell
does the same and later cells count what's in it.

`make_reservation` is the same function wrapped as a tool. Print its
description: this is what the model will read, and it says plainly that
calling the tool books. It doesn't say the tool asks anyone first, because
it doesn't.

In [18]:
from hotelbot.reservations import RESERVATIONS_FILE, add_reservation
from hotelbot.tools import make_reservation

add_reservation("lis-004", "olek", "2026-10-10", "2026-10-12")
print(RESERVATIONS_FILE.read_text())
reset_reservations()
print("after reset:", load_reservations())
print()
print(make_reservation.description)

[
  {
    "id": "res-0001",
    "hotel_id": "lis-004",
    "guest": "olek",
    "check_in": "2026-10-10",
    "check_out": "2026-10-12",
    "nights": 2,
    "total": 280.0,
    "created_at": "2026-10-01T13:23:22"
  }
]

after reset: []

Book a stay. Calling this writes a reservation immediately.

Args:
    hotel_id: The hotel's id, e.g. "lis-004".
    guest: The name the reservation is under.
    check_in: Check-in date, ISO format "YYYY-MM-DD".
    check_out: Check-out date, ISO format "YYYY-MM-DD", after check_in.

Returns:
    The written reservation as a dict with id, hotel_id, guest, check_in,
    check_out, nights, total, and created_at. If the stay isn't available,
    nothing is written and the dict is {"error": <reason>} instead. This
    never raises.


**What a model does with a pen.** Build an agent here with the same tools,
prompt and typed answer as `build_agent()`, plus `make_reservation`, and
nothing else in the way — no middleware, which the next cells introduce.
Then give it two messages, three times each, and count what was written.

The first only asks a price: "Casa do Castelo 10-12 Oct for olek looks good —
what would that come to?" Predict: does it book? The second asks it to book,
but with vague dates: "the second weekend of October". Predict: does it
check the dates with you before writing, or pick some and book?

Whatever the counts are, look at what decided them. If the first message
wrote nothing, that was the model's judgement on the wording, and a
different wording could tip it. If the second wrote three reservations, the
dates in them are dates the model picked and nobody saw before they were
written.

In [19]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model

from hotelbot.config import CHAT_MODEL
from hotelbot.models import BookingProposal, PlainAnswer
from hotelbot.prompts import get_prompt
from hotelbot.tools import check_availability, get_hotel, lookup_policy, search_hotels

pen_agent = create_agent(
    init_chat_model(CHAT_MODEL, reasoning_effort="low"),
    tools=[search_hotels, get_hotel, check_availability, lookup_policy, make_reservation],
    system_prompt=get_prompt("v1"),
    response_format=ToolStrategy(BookingProposal | PlainAnswer),
)

for _ in range(3):
    run(pen_agent, "Casa do Castelo 10-12 Oct for olek looks good — what would that come to?")
print("after the price question:", len(load_reservations()), "reservations")

for _ in range(3):
    run(pen_agent, "Book Casa do Castelo for olek, the second weekend of October.")
print("after the booking request:", len(load_reservations()), "reservations")
print()
for r in load_reservations():
    print(r.id, r.hotel_id, r.check_in, "to", r.check_out, f"€{r.total}")

reset_reservations()

after the price question: 0 reservations
after the booking request: 2 reservations

res-0001 lis-004 2026-10-09 to 2026-10-11 €280.0
res-0002 lis-004 2026-10-09 to 2026-10-11 €280.0


**The loop has hooks.** Book 2's loop was: call the model; if its reply has
tool calls, run them, append the results, and call the model again; stop
when a reply has none. A *middleware* is a class whose methods the loop calls
at fixed points along the way, so code can step in without rewriting the
loop. There are three points this book uses:

```
 ┌─▶ before_model      sees the messages about to go to the model
 │        │
 │    model call
 │        │
 │    after_model      sees the model's reply, before any tool runs
 │        │
 │    tool calls? ── no ──▶ final answer
 │        │ yes
 │    wrap_tool_call   wraps each tool run: code before, the tool, code after
 │        │
 └── tool results appended
```

A middleware implements only the methods it needs. The next cells use one
that lives in `after_model`, where it can stop the loop after the model has
asked for a booking but before the booking tool runs.

**The approval gate.** `build_agent()` now gives the agent `make_reservation`,
and with it `HumanInTheLoopMiddleware(interrupt_on={"make_reservation": True})`.
In `after_model`, it checks the model's reply for a `make_reservation` call.
If there is one, the run stops right there: the checkpointer saves the
thread, and the call returns without running the tool. This is why the
checkpointer had to exist first — a stopped run is only a saved thread
waiting to be continued.

`run()` puts what the gate is waiting on in `outcome.interrupt`: the tool's
name and the exact arguments it will run with if you say yes. Predict before
running: after this cell, how many reservations are in the file?

In [20]:
booking = "Book Casa do Castelo 10-12 Oct for olek."

paused = run(agent, booking)
request = paused.interrupt["action_requests"][0]

print("waiting on:", request["name"], request["args"])
print("answer so far:", paused.structured_response)
print("reservations:", load_reservations())

waiting on: make_reservation {'hotel_id': 'lis-004', 'guest': 'olek', 'check_in': '2026-10-10', 'check_out': '2026-10-12'}
answer so far: None
reservations: []


**Approve.** `resume(agent, decision, thread_id)` sends your decision to the
paused thread. The checkpointer loads the saved messages, the gate lets the
waiting `make_reservation` call through, the tool writes the reservation, and
the model reads the result and writes its answer — the loop carries on as if
it had never stopped. Predict the count this time.

This resumes the run from the cell above. To run this cell again, run that
one first: a thread can only be resumed while it is paused.

In [21]:
from hotelbot.agent import resume

approved = resume(agent, {"type": "approve"}, paused.thread_id)
print(approved.structured_response)
print()
for r in load_reservations():
    print(r.id, r.hotel_id, r.check_in, "to", r.check_out, f"€{r.total}")

text='Booked! Casa do Castelo, Lisbon, Oct 10–12 (2 nights) for Olek. Total: €280. Confirmation: res-0001.'

res-0001 lis-004 2026-10-10 to 2026-10-12 €280.0


**Edit.** Instead of yes or no, you can send the call back with different
arguments. Reset the file, make the same request, and take the arguments the
model proposed with `check_out` moved to `2026-10-13`. The tool runs with
your arguments, not the model's: three nights, not two.

The model only learns about the change from the tool's result, which comes
back after the tool has run. Read its answer — does it report the dates you
set, or the dates it asked for?

In [22]:
reset_reservations()

paused = run(agent, booking)
args = {**paused.interrupt["action_requests"][0]["args"], "check_out": "2026-10-13"}
print("sending back:", args)
print()

edited = resume(
    agent,
    {"type": "edit", "edited_action": {"name": "make_reservation", "args": args}},
    paused.thread_id,
)
print(edited.structured_response)
print()
for r in load_reservations():
    print(r.id, r.hotel_id, r.check_in, "to", r.check_out, f"€{r.total}")

sending back: {'hotel_id': 'lis-004', 'guest': 'olek', 'check_in': '2026-10-10', 'check_out': '2026-10-13'}

text="Booked! Casa do Castelo, Lisbon, for olek, check-in Oct 10 – check-out Oct 12... though I notice the confirmation shows check-out as Oct 13 (3 nights, €420 total) — please let me know if that's not what you intended."

res-0001 lis-004 2026-10-10 to 2026-10-13 €420.0


**Reject.** Saying no doesn't end the run with an error. The gate turns your
message into a `ToolMessage` in place of the tool's result — "the user
rejected this call, with this reason" — and calls the model again. The model
reads that like any other tool result and replies to it. Predict: what does
it say, and how many reservations are written?

In [23]:
reset_reservations()

paused = run(agent, booking)
rejected = resume(agent, {"type": "reject", "message": "not that one"}, paused.thread_id)

print(rejected.structured_response)
print("reservations:", load_reservations())
print()
for m in rejected.messages[-4:]:
    detail = [call["name"] for call in m.tool_calls] if getattr(m, "tool_calls", None) else m.text[:90]
    print(f"{type(m).__name__:<13} {detail}")

text="Okay, I won't book that. Could you clarify what you'd like instead?"
reservations: []

AIMessage     ['make_reservation']
ToolMessage   User rejected the tool call for `make_reservation` with reason: not that one
AIMessage     ['PlainAnswer']
ToolMessage   Returning structured response: text="Okay, I won't book that. Could you clarify what you'd


**Long conversations get summarised.** A thread only grows: every turn
appends your message, the model's tool calls, the tool results and the
answer, and every model call sends all of it again. Models measure text in
*tokens* — pieces of a few characters each, roughly four characters per token
in English — and both the price of a call and the most a model can read at
once are counted in tokens.

`SummarizationMiddleware` works in `before_model`. Before each model call it
checks the thread's size against `trigger=("tokens", 5500)`. The size it
checks includes the token total the API reported for the previous call,
which counts the system prompt and every tool's description too — about
2,500 tokens before you've said anything. Once the thread passes 5,500, it
sends the older messages to a cheaper model (`CHEAP_MODEL`) with a request
to summarise them, then replaces them with that one summary message.
`keep=("messages", 6)` leaves the last six messages as they were, so the
latest exchange is never summarised away.

Ten short questions on one thread, several of which pull a city's full hotel
list into the conversation. Each turn prints how many messages the thread
holds and the token total of its last model call. Predict roughly which turn
it fires on, and what happens to both numbers when it does.

In [24]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import AIMessage

from hotelbot.config import CHEAP_MODEL

summarising_agent = build_agent(
    middleware=[SummarizationMiddleware(model=CHEAP_MODEL, trigger=("tokens", 5500), keep=("messages", 6))]
)


def is_summary(message):
    return message.additional_kwargs.get("lc_source") == "summarization"


questions = [
    "Hi, I'm planning a trip in October.",
    "I'm travelling with my dog Bolt, and my budget is €200 a night.",
    "What hotels are there in Lisbon?",
    "And in Porto?",
    "And in Madrid?",
    "And in Seville?",
    "Which city had the cheapest option?",
    "Does that one allow pets?",
    "What's the cancellation policy?",
    "What time is check-in?",
]

for turn, question in enumerate(questions, start=1):
    outcome = run(summarising_agent, question, thread_id="long-trip")
    last_call = [m for m in outcome.messages if isinstance(m, AIMessage)][-1]
    summarised = any(is_summary(m) for m in outcome.messages)
    print(
        f"turn {turn:2}: {len(outcome.messages):3} messages, "
        f"last call {last_call.usage_metadata['total_tokens']:5} tokens, "
        f"summary in thread: {summarised}"
    )

turn  1:   3 messages, last call  2581 tokens, summary in thread: False
turn  2:   6 messages, last call  2765 tokens, summary in thread: False
turn  3:  11 messages, last call  3367 tokens, summary in thread: False
turn  4:  16 messages, last call  4227 tokens, summary in thread: False
turn  5:  21 messages, last call  4959 tokens, summary in thread: False
turn  6:  26 messages, last call  5805 tokens, summary in thread: False
turn  7:   9 messages, last call  3971 tokens, summary in thread: True
turn  8:  12 messages, last call  4128 tokens, summary in thread: True
turn  9:  17 messages, last call  4857 tokens, summary in thread: True
turn 10:  22 messages, last call  5569 tokens, summary in thread: True


**What the summary kept.** The dog's name and the budget were said once, in
turn 2, and turn 2 is no longer in the thread — only what the summary says
about it is. Print the summary message, then ask for both facts on the same
thread. Predict: does the answer still have them? Whatever the summary left
out, the agent no longer knows.

In [25]:
summary = next(m for m in outcome.messages if is_summary(m))
print(summary.text)
print()

recall = run(summarising_agent, "What's my dog's name, and what's my budget?", thread_id="long-trip")
print(recall.structured_response)

Here is a summary of the conversation to date:

## SESSION INTENT

Plan and book a pet-friendly hotel stay in October for the user and their dog, Bolt, with a budget of €200 per night maximum.

## SUMMARY

User is traveling with their dog Bolt in October with a budget constraint of €200/night and requires pet-friendly accommodations. User explored hotel options across three cities:

**Lisbon Options (Both available all October):**
- Principe Real Boutique: €178/night, rating 4.0, amenities: spa, pets, ac
- Graca Residence: €200/night, rating 4.7, amenities: pets, ac, elevator

**Porto Options (All available all October):**
- Foz do Douro Palace: €136/night, rating 4.4, amenities: parking, spa, pets
- Bonfim Boutique: €158/night, rating 4.0, amenities: spa, pets, ac
- Aliados Residence: €180/night, rating 4.7, amenities: pets, ac, elevator

**Madrid Options (Both available all October):**
- Chueca Palace: €166/night, rating 4.4, amenities: parking, spa, pets
- Retiro Boutique: €188/nigh

**Your own hook.** `hotelbot.middleware.LogToolCalls` is a middleware written
for this project, and it is short enough to read whole — the cell prints its
source. It implements one method, `wrap_tool_call(request, handler)`.
`request.tool_call` is the call the model asked for, with its name and
arguments. `handler` is a function that runs the tool and returns its
`ToolMessage`. Code before `handler(request)` runs before the tool, code after
it runs after, and whatever the method returns is the result the model sees.
This one prints and passes the result through unchanged.

Build an agent with it and ask for a booking. Predict: when the run pauses,
does `make_reservation` appear in the log yet? And after you approve?
`build_agent()` puts the approval gate in front of whatever middleware you
pass, so passing your own doesn't remove it.

In [26]:
import inspect

from hotelbot.middleware import LogToolCalls

print(inspect.getsource(LogToolCalls))

reset_reservations()
logged_agent = build_agent(middleware=[LogToolCalls()])

paused = run(logged_agent, booking)
print("paused on:", paused.interrupt["action_requests"][0]["name"], "| reservations:", len(load_reservations()))
print()

approved = resume(logged_agent, {"type": "approve"}, paused.thread_id)
print("reservations:", len(load_reservations()))

reset_reservations()

class LogToolCalls(AgentMiddleware):
    """Print every tool call the agent makes: the tool's name and arguments
    before it runs, and how many characters it returned after.

    It only prints; it never changes a call or its result.
    """

    def wrap_tool_call(self, request, handler):
        call = request.tool_call
        print(f"-> {call['name']} {call['args']}")
        result = handler(request)
        print(f"<- {call['name']} returned {len(str(getattr(result, 'content', '')))} characters")
        return result

-> search_hotels {'city': 'Lisbon'}
<- search_hotels returned 2454 characters
paused on: make_reservation | reservations: 0

-> make_reservation {'hotel_id': 'lis-004', 'guest': 'olek', 'check_in': '2026-10-10', 'check_out': '2026-10-12'}
<- make_reservation returned 177 characters
reservations: 1


**What you can do now.** The agent holds a conversation: a thread id keeps
the messages, and a long thread is summarised rather than resent in full. It
can book a room, and it can't book one without you — the approval gate
pauses every `make_reservation` call until you approve, edit or reject it,
whatever other middleware is added. And you can step into the loop yourself
at `before_model`, `after_model` or around a tool call.

Every book so far has used the same short system prompt, written once in
book 3 and never tested. Book 6 writes three versions of it, sends the same
five requests through each, and compares what each
version got right.